In [1]:
import time
from ip import *
import numpy as np
import networkx as nx
from scipy import ndimage
import matplotlib.pyplot as plt
from skimage.filters import threshold_triangle
from skimage.util import img_as_ubyte, img_as_float

In [2]:
volumeOP1 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_1"))
volumeOP2 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_2"))
volumeOP3 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_3"))
volumeOP4 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_4"))
volumeOP5 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_5"))
volumeOP6 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_6"))
volumeOP7 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_7"))
volumeOP8 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_8"))
volumeOP9 = img_as_float(load_tif_stack("./OlfactoryProjectionFibers/ImageStacks/OP_9"))

In [3]:
rootOP_1 = (30.979,429.04,0)
rootOP_2 = (0.72501,391.08,25)
rootOP_3 = (93.742,179,38)
rootOP_4 = (128.2,504.37,0.3)
rootOP_5 = (185.7,264.02,33)
rootOP_6 = (15.074,412.01,10)
rootOP_7 = (119.76,215.98,39)
rootOP_8 = (118.64,181.34,55)
rootOP_9 = (64.56,364.47,4)

In [4]:
vol3d = img_as_ubyte(volumeOP1.copy())
curr_root = rootOP_1

In [ ]:
kernel1 = np.array(
    [[1 / 16, 1 / 8, 1 / 16], [1 / 8, 1 / 4, 1 / 8], [1 / 16, 1 / 8, 1 / 16]]
)

kernel1 = kernel1 / np.sum(kernel1)
kernel2 = np.array(
    [
        [1 / 256, 1 / 64, 3 / 128, 1 / 64, 1 / 256],
        [1 / 64, 1 / 16, 3 / 32, 1 / 16, 1 / 64],
        [3 / 128, 3 / 32, 9 / 64, 3 / 32, 3 / 128],
        [1 / 64, 1 / 16, 3 / 32, 1 / 16, 1 / 64],
        [1 / 256, 1 / 64, 3 / 128, 1 / 64, 1 / 256],
    ]
)

kernel2 = kernel2 / np.sum(kernel2)

In [ ]:
image3d = convolve3d(vol3d, kernel1)

In [ ]:
thresh_tri = threshold_triangle(image3d)
binary_tri = img_as_ubyte(image3d > thresh_tri)
seg2 = img_as_ubyte(segment(vol3d, binary_tri))

In [ ]:
sigma_scale_space1 = np.linspace(5, 75, 5)
sigma_scale_space2 = np.linspace(35, 35, 1)
print(sigma_scale_space1, sigma_scale_space2)

In [ ]:
# Create all parameters beforehand
params = [
    (image3d, "canny", 8, sigma_scale_space1, 4, 4),
    (vol3d, "canny", 8, sigma_scale_space1, 4, 4),
    (seg2, "canny", 8, sigma_scale_space1, 4, 4),
]

In [ ]:
vfc_result1 = scale_space_medialness(*params[0])

In [ ]:
vfc_result2 = scale_space_medialness(*params[1])
vfc_result2 = convolve3d(vfc_result2, kernel1)

In [ ]:
vfc_result3 = scale_space_medialness(*params[2])
vfc_result3 = convolve3d(vfc_result3, kernel1)

In [ ]:
# simple_imshow(
#     [
#         projection2d(img_as_ubyte(vfc_result1), "min"),
#         projection2d(img_as_ubyte(vfc_result2), "min"),
#         projection2d(img_as_ubyte(vfc_result3), "min"),
#     ]
# )

In [ ]:
local_maxima_coords1, _ = local_maxima_3D(vfc_result1, 1)
local_maxima_coords2, _ = local_maxima_3D(vfc_result2, 1)
local_maxima_coords3, _ = local_maxima_3D(vfc_result3, 1)

In [ ]:
test_img1 = create_maxima_image(local_maxima_coords1, image3d.shape)
test_img2 = create_maxima_image(local_maxima_coords2, image3d.shape)
test_img3 = create_maxima_image(local_maxima_coords3, image3d.shape)

In [ ]:
# download(test_img1, "./Test/test_img1")
# download(test_img2, "./Test/test_img2")
# download(test_img3, "./Test/test_img3")

In [ ]:
simple_imshow(
    [
        projection2d(test_img1, "max"),
        projection2d(test_img2, "max"),
        projection2d(test_img3, "max"),
    ]
)

In [ ]:
graph1 = create_graph_from_kdtree(local_maxima_coords1, 20, 30, 2)
graph2 = create_graph_from_kdtree(local_maxima_coords2, 20, 30, 2)
graph3 = create_graph_from_kdtree(local_maxima_coords3, 20, 30, 2)

In [ ]:
print(
    f"graph1 -> nodes:{graph1.number_of_nodes()} | edges:{graph1.number_of_edges()}\ngraph2 -> nodes:{graph2.number_of_nodes()} | edges:{graph2.number_of_edges()}\ngraph3 -> nodes:{graph3.number_of_nodes()} | edges:{graph3.number_of_edges()}"
)

In [ ]:
root1 = find_nearest_foreground_point(test_img1, curr_root)
root2 = find_nearest_foreground_point(test_img2, curr_root)
root3 = find_nearest_foreground_point(test_img3, curr_root)
print(root1, root2, root3)

root1_idx = get_node_index_by_position(graph1, root1)
root2_idx = get_node_index_by_position(graph2, root2) 
root3_idx = get_node_index_by_position(graph3, root3) 
print(root1_idx, root2_idx, root3_idx)

In [ ]:
graph_filtered1 = filter_graph_by_shortest_paths(graph1, root1_idx)
graph_filtered2 = filter_graph_by_length(graph1, 30)
# graph_filtered3 = filter_graph_by_shortest_paths(graph3, root3_idx)

# filter_graph_by_shortest_paths(graph2, root2)

In [ ]:
# print(
#     f"graph_filtered1 -> nodes:{graph_filtered1.number_of_nodes()} | edges:{graph_filtered1.number_of_edges()}\ngraph_filtered2 -> nodes:{graph_filtered2.number_of_nodes()} | edges:{graph_filtered2.number_of_edges()}\ngraph_filtered3 -> nodes:{graph_filtered3.number_of_nodes()} | edges:{graph_filtered3.number_of_edges()}"
# )

melhorar algoritmo de geracao de $grafo/agm$

DEZ 04 - implementacao de outros modos para a geracao do Edge Map, canny, laplacian, prewitt

Canny se saiu MUITO BEM em comparacao aos outros

In [ ]:
visualize_medial_graph(vol3d, graph1)
# visualize_medial_graph(vol3d, graph_filtered1)

In [ ]:
visualize_medial_graph(vol3d, graph_filtered1)
visualize_medial_graph(vol3d, graph_filtered2)

In [ ]:
# visualize_medial_graph(vol3d, graph3)
# visualize_medial_graph(vol3d, graph_filtered3)

In [ ]:
mst_subgraph = nx.minimum_spanning_tree(graph1)


In [ ]:
visualize_medial_graph(vol3d, mst_subgraph)